## 0. Nạp dữ liệu thô ban đầu từ Bronze

In [29]:
import os
import json
import re
from datetime import datetime
import pandas as pd

# 1. Đọc dữ liệu từ bronze
file_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
if not os.path.exists(file_path):
    file_path = os.path.abspath('data/bronze/bronze_details_valid.json')

with open(file_path, 'r', encoding='utf-8') as f:
    bronze_data = json.load(f)

# 2. Trích xuất bảng dữ liệu thô ban đầu
records = []
for item in bronze_data:
    desc_parts = [item.get(k, '') for k in ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw'] if item.get(k)]
    records.append({
        'job_id': item.get('detail_id') or f"{item.get('source', '')}_{item.get('source_job_id', '')}",
        'company_name_raw': (item.get('company_name_raw') or '').strip(),
        'title_raw': (item.get('title_raw') or '').strip(),
        'description_raw': '\n\n'.join(desc_parts),
        'required_skills_raw': (item.get('required_skills_raw') or '').strip(),
        'preferred_skills_raw': (item.get('preferred_skills_raw') or '').strip(),
        'experience_raw': (item.get('experience_raw') or '').strip(),
        'location_raw': (item.get('location_raw') or item.get('company_location_raw') or '').strip(),
        'salary_raw': (item.get('salary_raw') or '').strip(),
        'employment_type_raw': (item.get('employment_type_raw') or '').strip(),
        'work_mode_raw': (item.get('work_mode_raw') or '').strip(),
        'deadline_raw': (item.get('deadline_raw') or '').strip()
    })

jobs = pd.DataFrame(records)
print(f"Đã nạp {len(jobs):,} bản ghi ban đầu.")
jobs.head(3)


Đã nạp 3,997 bản ghi ban đầu.


,job_id,company_name_raw,title_raw,description_raw,required_skills_raw,preferred_skills_raw,experience_raw,location_raw,salary_raw,employment_type_raw,work_mode_raw,deadline_raw
0,topcv_2314643,Công ty TNHH ZIGExN VeNtura,Full-Stack Web Engineer Intern/Fresher (AI Co...,COMPANY INTRODUCTION\nAbout our parent company...,"Không yêu cầu kinh nghiệm chuyên môn, Fullstac...",,Không yêu cầu,"Floor 11, Vina Building, 131 Xo Viet Nghe Tinh...",Thoả thuận,Thực tập sinh,Onsite,2026-10-28T23:59:59+07:00
1,topcv_2314547,Công ty TNHH Điện tử R&P,Kỹ Sư Hệ Thống Nhúng,"R&P là công ty công nghệ, hoạt động trên lĩnh ...","Không yêu cầu kinh nghiệm chuyên môn, Embedded...",,Không yêu cầu,"65 Lê Trung Nghĩa, Phường Bảy Hiền, Hồ Chí Minh",Thoả thuận,Nhân viên,Onsite,2026-10-28T23:59:59+07:00
2,topcv_2321616,CÔNG TY TNHH FOXSCORE,UI/UX Designer - Mobile App- [HCM] - Thu Nhập ...,FOXSCORE – NỀN TẢNG THỂ THAO THÔNG MINH\nRa đờ...,"2 năm kinh nghiệm chuyên môn, Product Owner/Pr...",,2 năm,"62/6-8-10 Trương Công Định, Phường Tân Bình, H...",15000000,Nhân viên,Onsite,2026-11-05T23:59:59+07:00


## Bước 1: Làm sạch Tên công ty (`company_name_raw` ➔ `company_name`)

Loại bỏ các tiền tố/hậu tố pháp lý (*Công ty TNHH, Cổ phần, JSC, LLC, Co., Ltd...*) để đồng nhất tên thương hiệu.

In [30]:
def clean_company_name(name):
    if not name or pd.isna(name):
        return "Không rõ"
    s = str(name).strip()
    s = re.sub(r'^(?:công ty|cty)?\s*(?:tnhh mtv|tnhh|cổ phần|cp|tập đoàn)\s*', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[\s,\-]+(?:tnhh|cổ phần|jsc|llc|inc|corp|co\.,?\s*ltd\.?|ltd\.?)$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[\s,\-]+(?:chi nhánh|vietnam|việt nam)$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+', ' ', s).strip(' -,/:;')
    return s if s else str(name).strip()

jobs['company_name'] = jobs['company_name_raw'].apply(clean_company_name)

print("=== BƯỚC 1: TRƯỚC VÀ SAU KHI LÀM SẠCH COMPANY NAME ===")
jobs[['company_name_raw', 'company_name']].head(5)


=== BƯỚC 1: TRƯỚC VÀ SAU KHI LÀM SẠCH COMPANY NAME ===


,company_name_raw,company_name
0,Công ty TNHH ZIGExN VeNtura,ZIGExN VeNtura
1,Công ty TNHH Điện tử R&P,Điện tử R&P
2,CÔNG TY TNHH FOXSCORE,FOXSCORE
3,CÔNG TY CỔ PHẦN THƯƠNG MẠI QUỐC TẾ LB GLOBAL,THƯƠNG MẠI QUỐC TẾ LB GLOBAL
4,CÔNG TY TNHH VODAPLAY VIỆT NAM,VODAPLAY


## Bước 2: Làm sạch Tiêu đề việc làm (`title_raw` ➔ `title`)

Loại bỏ nội dung trong ngoặc vuông `[...]`, địa điểm `(HCM)`, tiền tố tuyển gấp, và các đoạn quảng cáo lương ở đuôi.

In [31]:
def clean_job_title(title):
    if not title or pd.isna(title):
        return ""
    t = str(title).strip()
    t = re.sub(r'\[.*?\]', '', t)
    t = re.sub(r'\((?:hà nội|hcm|đà nẵng|hồ chí minh|toàn quốc)\)', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\s*(?:tuyển gấp|tuyển dụng|gấp:?)\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'(?:-|\|)?\s*(?:thu nhập|lương|up to|upto|lên đến|tới)\s*.*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'(?:-|\|)?\s*(?:hấp dẫn|deal|thỏa thuận|cạnh tranh)\s*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip(' -|/,:;')
    return t

jobs['title'] = jobs['title_raw'].apply(clean_job_title)

print("=== BƯỚC 2: TRƯỚC VÀ SAU KHI LÀM SẠCH TITLE ===")
jobs[['title_raw', 'title']].head(5)


=== BƯỚC 2: TRƯỚC VÀ SAU KHI LÀM SẠCH TITLE ===


,title_raw,title
0,Full-Stack Web Engineer Intern/Fresher (AI Co...,Full-Stack Web Engineer Intern/Fresher (AI Cod...
1,Kỹ Sư Hệ Thống Nhúng,Kỹ Sư Hệ Thống Nhúng
2,UI/UX Designer - Mobile App- [HCM] - Thu Nhập ...,UI/UX Designer - Mobile App
3,Business Analyst - Data Hệ Thống Phân Phối - C...,Business Analyst - Data Hệ Thống Phân Phối - C...
4,Junior QA Tester (Manual Testing) Upto 16M,Junior QA Tester (Manual Testing)


## Bước 3: Làm sạch Văn bản Mô tả (`description_raw` ➔ `description_clean`)

Gỡ thẻ HTML, giải mã thực thể HTML (`&nbsp;`, `&amp;`), đồng bộ bullet points về `- `, và bỏ các dòng trống thừa.

In [32]:
def clean_html_description(text):
    if not text or pd.isna(text):
        return ""
    s = str(text)
    s = s.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>').replace('&quot;', '"')
    s = re.sub(r'<(?:br|/p|/li|/div|/h\d)[^>]*>', '\n', s, flags=re.IGNORECASE)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'^[ \t]*[-\*\+\–•][ \t]*', '- ', s, flags=re.MULTILINE)
    lines = [re.sub(r'[ \t]+', ' ', l).strip() for l in s.split('\n')]
    return '\n'.join([l for l in lines if l])

jobs['description_clean'] = jobs['description_raw'].apply(clean_html_description)

print("=== BƯỚC 3: MẪU DESCRIPTION_CLEAN SAU KHI LÀM SẠCH (400 KÝ TỰ ĐẦU) ===")
print(jobs['description_clean'].iloc[0][:400] + "...")
jobs[['title', 'description_clean']].head(3)


=== BƯỚC 3: MẪU DESCRIPTION_CLEAN SAU KHI LÀM SẠCH (400 KÝ TỰ ĐẦU) ===
COMPANY INTRODUCTION
About our parent company: ZIGExN Co.,Ltd
- ZIGEXN is a leading life media platform company in Japan, offering over 40 services in various life event domains such as jobs, housing, cars, and travel.
- ZIGExN, the company's flagship, received the Japan Technology Fast 50 award from Deloitte in 2015 and has been recognized as a Great Place to Work in Japan from 2011 to 2016.
- ZI...


,title,description_clean
0,Full-Stack Web Engineer Intern/Fresher (AI Cod...,COMPANY INTRODUCTION\nAbout our parent company...
1,Kỹ Sư Hệ Thống Nhúng,"R&P là công ty công nghệ, hoạt động trên lĩnh ..."
2,UI/UX Designer - Mobile App,FOXSCORE – NỀN TẢNG THỂ THAO THÔNG MINH\nRa đờ...


## Bước 4: Làm sạch Kỹ năng yêu cầu (`required_skills_raw` ➔ `required_skills`)

Tách chuỗi theo dấu phẩy và loại bỏ các từ dừng tuyển dụng rác (*"Đại Học trở lên"*, *"Không yêu cầu kinh nghiệm"*, *"Nhân viên"*).

In [33]:
SKILL_STOPWORDS = {
    'đại học trở lên', 'cao đẳng trở lên', 'không yêu cầu kinh nghiệm', 'không yêu cầu kinh nghiệm chuyên môn',
    'chưa có kinh nghiệm', '1 năm kinh nghiệm chuyên môn', '2 năm kinh nghiệm chuyên môn', 'dưới 1 năm kinh nghiệm chuyên môn',
    'nhân viên', 'thực tập sinh', 'toàn thời gian', 'bán thời gian', 'it - phần mềm', 'công nghệ thông tin'
}

def clean_required_skills(skills_str):
    if not skills_str or pd.isna(skills_str):
        return []
    # Tách theo các ký tự phân cách
    parts = re.split(r'[,;|/•\n]', str(skills_str))
    clean_list = []
    seen = set()
    for p in parts:
        item = re.sub(r'\s+', ' ', p).strip()
        if item and len(item) > 1 and item.lower() not in SKILL_STOPWORDS and item.lower() not in seen:
            seen.add(item.lower())
            clean_list.append(item)
    return clean_list

jobs['required_skills'] = jobs['required_skills_raw'].apply(clean_required_skills)

print("=== BƯỚC 4: TRƯỚC VÀ SAU KHI LÀM SẠCH REQUIRED_SKILLS ===")
jobs[['required_skills_raw', 'required_skills']].head(5)


=== BƯỚC 4: TRƯỚC VÀ SAU KHI LÀM SẠCH REQUIRED_SKILLS ===


,required_skills_raw,required_skills
0,"Không yêu cầu kinh nghiệm chuyên môn, Fullstac...","[Fullstack Developer, Tiếng Anh Đọc hiểu, Tiến..."
1,"Không yêu cầu kinh nghiệm chuyên môn, Embedded...","[Embedded Engineer, Lập trình nhúng, Điện, Điệ..."
2,"2 năm kinh nghiệm chuyên môn, Product Owner/Pr...","[Product Owner, Product Manager, Game, Bảo hiể..."
3,"Dưới 1 năm kinh nghiệm chuyên môn, Vận hành sà...",[Vận hành sàn thương mại điện tử]
4,"1 năm kinh nghiệm chuyên môn, Manual Tester, I...","[Manual Tester, Bảo hiểm xã hội, Bảo hiểm sức ..."


## Bước 5: Bóc tách Kỹ năng ưu tiên (`preferred_skills_raw` & `description_clean` ➔ `preferred_skills`)

Do trên các sàn trường này bị rỗng 100%, ta sẽ quét đoạn văn bản `description_clean` tìm các đoạn có chứa từ khóa *"Ưu tiên"*, *"Điểm cộng"*, *"Nice to have"* để trích xuất kỹ năng.

In [34]:
POPULAR_TECH_KEYWORDS = [
    'Docker', 'Kubernetes', 'AWS', 'GCP', 'Azure', 'CI/CD', 'Git', 'Redis', 'Kafka', 'GraphQL',
    'React', 'Vue', 'Angular', 'Node.js', 'Python', 'Java', 'Golang', 'TypeScript', 'Flutter',
    'PostgreSQL', 'MongoDB', 'MySQL', 'Elasticsearch', 'Spring Boot', 'Next.js', 'TailwindCSS'
]

def extract_preferred_skills(row):
    raw_pref = row['preferred_skills_raw']
    if raw_pref and pd.notnull(raw_pref) and str(raw_pref).strip():
        return clean_required_skills(raw_pref)
    
    desc = str(row['description_clean']).lower()
    pref_section = ""
    m = re.search(r'(?:ưu tiên|điểm cộng|nice to have|plus|yêu cầu khác)[\s:]*([^\.\n]+)', desc)
    if m:
        pref_section = m.group(1)
            
    found_skills = []
    req_skills_lower = [k.lower() for k in row['required_skills']]
    for tech in POPULAR_TECH_KEYWORDS:
        if tech.lower() in pref_section and tech.lower() not in req_skills_lower:
            found_skills.append(tech)
    return found_skills

jobs['preferred_skills'] = jobs.apply(extract_preferred_skills, axis=1)

print("=== BƯỚC 5: KẾT QUẢ BÓC TÁCH PREFERRED_SKILLS ===")
jobs[['title', 'required_skills', 'preferred_skills']].head(5)


=== BƯỚC 5: KẾT QUẢ BÓC TÁCH PREFERRED_SKILLS ===


,title,required_skills,preferred_skills
0,Full-Stack Web Engineer Intern/Fresher (AI Cod...,"[Fullstack Developer, Tiếng Anh Đọc hiểu, Tiến...",[]
1,Kỹ Sư Hệ Thống Nhúng,"[Embedded Engineer, Lập trình nhúng, Điện, Điệ...",[]
2,UI/UX Designer - Mobile App,"[Product Owner, Product Manager, Game, Bảo hiể...",[]
3,Business Analyst - Data Hệ Thống Phân Phối - C...,[Vận hành sàn thương mại điện tử],[]
4,Junior QA Tester (Manual Testing),"[Manual Tester, Bảo hiểm xã hội, Bảo hiểm sức ...",[]


## Bước 6: Bóc tách Yêu cầu kinh nghiệm (`experience_raw` ➔ `experience_min`, `experience_max`)

Quy đổi các chuỗi tiếng Việt thô (*"Không yêu cầu"*, *"1 - 3 năm"*, *"Trên 5 năm"*, *"6 tháng"*) thành số năm dạng số thực.

In [35]:
def parse_experience_range(exp_str):
    if not exp_str or pd.isna(exp_str):
        return None, None
    s = str(exp_str).strip().lower()
    if any(k in s for k in ['không yêu cầu', 'chưa có kinh nghiệm', 'không cần', 'fresher', 'intern']):
        return 0.0, 0.0
    m_range = re.search(r'(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*năm', s)
    if m_range:
        return float(m_range.group(1)), float(m_range.group(2))
    m_min = re.search(r'(?:trên|từ|tối thiểu|\+)\s*(\d+(?:\.\d+)?)\s*năm|(\d+(?:\.\d+)?)\s*(?:\+|năm trở lên)', s)
    if m_min:
        v = float(m_min.group(1) or m_min.group(2))
        return v, None
    m_max = re.search(r'dưới\s*(\d+(?:\.\d+)?)\s*năm', s)
    if m_max:
        return 0.0, float(m_max.group(1))
    m_month = re.search(r'(\d+)\s*tháng', s)
    if m_month:
        years = round(float(m_month.group(1)) / 12.0, 1)
        return years, years
    m_exact = re.search(r'(\d+(?:\.\d+)?)\s*năm', s)
    if m_exact:
        v = float(m_exact.group(1))
        return v, v
    return None, None

exp_res = jobs['experience_raw'].apply(parse_experience_range)
jobs['experience_min'] = exp_res.apply(lambda x: x[0])
jobs['experience_max'] = exp_res.apply(lambda x: x[1])

print("=== BƯỚC 6: TRƯỚC VÀ SAU KHI BÓC TÁCH KINH NGHIỆM ===")
jobs[['experience_raw', 'experience_min', 'experience_max']].head(5)


=== BƯỚC 6: TRƯỚC VÀ SAU KHI BÓC TÁCH KINH NGHIỆM ===


,experience_raw,experience_min,experience_max
0,Không yêu cầu,0.0,0.0
1,Không yêu cầu,0.0,0.0
2,2 năm,2.0,2.0
3,3 tháng,0.2,0.2
4,1 năm,1.0,1.0


## Bước 7: Làm sạch Địa điểm & Trích xuất Thành phố (`location_raw` ➔ `location`, `city`)

Giữ địa chỉ chi tiết làm sạch, đồng thời gán nhãn thành phố chuẩn hóa (`Hồ Chí Minh`, `Hà Nội`, `Đà Nẵng`, `Khác`) phục vụ bộ lọc tìm việc.

In [36]:
def extract_city(loc_str):
    if not loc_str or pd.isna(loc_str):
        return "Toàn quốc"
    s = str(loc_str).lower()
    if any(k in s for k in ['hồ chí minh', 'hcm', 'sài gòn', 'saigon', 'thủ đức']):
        return "Hồ Chí Minh"
    if any(k in s for k in ['hà nội', 'hn', 'cầu giấy', 'nam từ liêm', 'ba đình']):
        return "Hà Nội"
    if any(k in s for k in ['đà nẵng', 'da nang']):
        return "Đà Nẵng"
    if any(k in s for k in ['cần thơ', 'hải phòng', 'bình dương', 'đồng nai']):
        return "Khác"
    return "Khác"

jobs['location'] = jobs['location_raw'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip() if pd.notnull(x) else "")
jobs['city'] = jobs['location'].apply(extract_city)

print("=== BƯỚC 7: TRƯỚC VÀ SAU KHI LÀM SẠCH LOCATION & TRÍCH XUẤT CITY ===")
jobs[['location_raw', 'location', 'city']].head(5)


=== BƯỚC 7: TRƯỚC VÀ SAU KHI LÀM SẠCH LOCATION & TRÍCH XUẤT CITY ===


,location_raw,location,city
0,"Floor 11, Vina Building, 131 Xo Viet Nghe Tinh...","Floor 11, Vina Building, 131 Xo Viet Nghe Tinh...",Hồ Chí Minh
1,"65 Lê Trung Nghĩa, Phường Bảy Hiền, Hồ Chí Minh","65 Lê Trung Nghĩa, Phường Bảy Hiền, Hồ Chí Minh",Hồ Chí Minh
2,"62/6-8-10 Trương Công Định, Phường Tân Bình, H...","62/6-8-10 Trương Công Định, Phường Tân Bình, H...",Hồ Chí Minh
3,"Phường Cầu Giấy, Hà Nội","Phường Cầu Giấy, Hà Nội",Hà Nội
4,"Phường Cầu Giấy, Hà Nội","Phường Cầu Giấy, Hà Nội",Hà Nội


## Bước 8: Chuẩn hóa Mức lương (`salary_raw` ➔ `salary_min`, `salary_max` theo VNĐ/tháng)

Quy đổi tiền tệ (USD ➔ VNĐ tỷ giá 25,400; Triệu đồng ➔ VNĐ nhân 1,000,000), phân tách cận dưới và cận trên.

In [37]:
EXCHANGE_RATE_USD = 25400

def parse_salary_vnd(sal_str):
    if not sal_str or pd.isna(sal_str):
        return None, None
    s = str(sal_str).strip().lower()
    if any(k in s for k in ['thương lượng', 'cạnh tranh', 'thoa thuan', 'thỏa thuận', 'negotiable', 'deal']):
        return None, None
    
    has_usd = ('$' in s or 'usd' in s) and not ('tr' in s or 'triệu' in s)
    nums = [float(n.replace(',', '')) for n in re.findall(r'(\d+(?:,\d+)*(?:\.\d+)?)', s)]
    if not nums:
        return None, None
        
    if has_usd:
        nums = [n * EXCHANGE_RATE_USD for n in nums]
    else:
        if any(k in s for k in ['tr', 'triệu', 'trieu', 'm']) or any(n < 500 for n in nums):
            nums = [n * 1_000_000 for n in nums]
            
    if len(nums) >= 2:
        return min(nums[0], nums[1]), max(nums[0], nums[1])
    elif len(nums) == 1:
        val = nums[0]
        if any(k in s for k in ['tới', 'up to', 'dưới', '<', 'tối đa']):
            return None, val
        elif any(k in s for k in ['từ', 'trên', '>=', 'tối thiểu', 'from', '>']):
            return val, None
        return val, val
    return None, None

sal_res = jobs['salary_raw'].apply(parse_salary_vnd)
jobs['salary_min'] = sal_res.apply(lambda x: x[0])
jobs['salary_max'] = sal_res.apply(lambda x: x[1])

print("=== BƯỚC 8: TRƯỚC VÀ SAU KHI CHUẨN HÓA LƯƠNG SANG VNĐ ===")
jobs[['salary_raw', 'salary_min', 'salary_max']].head(5)


=== BƯỚC 8: TRƯỚC VÀ SAU KHI CHUẨN HÓA LƯƠNG SANG VNĐ ===


,salary_raw,salary_min,salary_max
0,Thoả thuận,NaN,NaN
1,Thoả thuận,NaN,NaN
2,15000000,15000000.0,15000000.0
3,12000000,12000000.0,12000000.0
4,Tới 16 triệu,NaN,16000000.0


## Bước 9: Chuẩn hóa Hình thức làm việc (`employment_type_raw` ➔ `employment_type`)

Ánh xạ về 4 nhóm chuẩn: `Intern`, `Part-time`, `Contract`, `Full-time`.

In [38]:
def clean_employment_type(emp_str, title_str=""):
    c = (str(emp_str) + " " + str(title_str)).lower()
    if any(k in c for k in ['intern', 'thực tập', 'trainee']):
        return 'Intern'
    if any(k in c for k in ['part-time', 'bán thời gian', 'parttime']):
        return 'Part-time'
    if any(k in c for k in ['freelance', 'hợp đồng', 'contract']):
        return 'Contract'
    return 'Full-time'

jobs['employment_type'] = jobs.apply(lambda r: clean_employment_type(r['employment_type_raw'], r['title']), axis=1)

print("=== BƯỚC 9: TRƯỚC VÀ SAU KHI CHUẨN HÓA EMPLOYMENT_TYPE ===")
jobs[['employment_type_raw', 'employment_type']].head(5)


=== BƯỚC 9: TRƯỚC VÀ SAU KHI CHUẨN HÓA EMPLOYMENT_TYPE ===


,employment_type_raw,employment_type
0,Thực tập sinh,Intern
1,Nhân viên,Full-time
2,Nhân viên,Full-time
3,Nhân viên,Full-time
4,Nhân viên,Full-time


## Bước 10: Chuẩn hóa Mô hình làm việc (`work_mode_raw` ➔ `work_mode`)

Ánh xạ về 3 nhóm chuẩn duy nhất: `Onsite`, `Hybrid`, `Remote`.

In [39]:
def clean_work_mode(mode_str, loc_str=""):
    c = (str(mode_str) + ' ' + str(loc_str)).lower()
    if any(k in c for k in ['remote', 'từ xa', 'làm việc từ xa']):
        return 'Remote'
    if any(k in c for k in ['hybrid', 'linh hoạt']):
        return 'Hybrid'
    return 'Onsite'

jobs['work_mode'] = jobs.apply(lambda r: clean_work_mode(r['work_mode_raw'], r['location']), axis=1)

print("=== BƯỚC 10: TRƯỚC VÀ SAU KHI CHUẨN HÓA WORK_MODE ===")
jobs[['work_mode_raw', 'work_mode']].head(5)


=== BƯỚC 10: TRƯỚC VÀ SAU KHI CHUẨN HÓA WORK_MODE ===


,work_mode_raw,work_mode
0,Onsite,Onsite
1,Onsite,Onsite
2,Onsite,Onsite
3,Onsite,Onsite
4,Onsite,Onsite


## Bước 11: Chuẩn hóa Hạn nộp & Tính hiệu lực (`deadline_raw` ➔ `deadline`, `is_active`)

Cắt lấy ngày `YYYY-MM-DD` và so sánh với ngày hiện tại để xác định tin còn hiệu lực (`is_active = True/False`).

In [40]:
def parse_deadline(d_str):
    if not d_str or pd.isna(d_str):
        return None, True
    s = str(d_str).strip()
    try:
        if 'T' in s:
            dt = datetime.fromisoformat(s)
        else:
            dt = datetime.strptime(s[:10], '%Y-%m-%d')
        d_formatted = dt.strftime('%Y-%m-%d')
        is_act = dt.date() >= datetime.now().date()
        return d_formatted, is_act
    except Exception:
        return s[:10], True

dl_res = jobs['deadline_raw'].apply(parse_deadline)
jobs['deadline'] = dl_res.apply(lambda x: x[0])
jobs['is_active'] = dl_res.apply(lambda x: x[1])

print("=== BƯỚC 11: TRƯỚC VÀ SAU KHI CHUẨN HÓA DEADLINE & IS_ACTIVE ===")
jobs[['deadline_raw', 'deadline', 'is_active']].head(5)


=== BƯỚC 11: TRƯỚC VÀ SAU KHI CHUẨN HÓA DEADLINE & IS_ACTIVE ===


,deadline_raw,deadline,is_active
0,2026-10-28T23:59:59+07:00,2026-10-28,True
1,2026-10-28T23:59:59+07:00,2026-10-28,True
2,2026-11-05T23:59:59+07:00,2026-11-05,True
3,2026-10-28T23:59:59+07:00,2026-10-28,True
4,2026-10-29T23:59:59+07:00,2026-10-29,True


## Bước 12: Khử trùng lặp thực sự (Deduplication theo Công Ty + Chức Danh)

Loại bỏ các tin trùng lặp cùng công ty đăng lại bài, ưu tiên giữ lại tin có **hạn chót xa nhất (mới nhất)** và **nội dung mô tả dài nhất**.

In [41]:
old_count = len(jobs)
jobs['desc_len'] = jobs['description_clean'].str.len()

jobs = jobs.sort_values(by=['company_name', 'title', 'deadline', 'desc_len'], ascending=[True, True, False, False])
jobs = jobs.drop_duplicates(subset=['company_name', 'title'], keep='first')
jobs = jobs.drop(columns=['desc_len'])
new_count = len(jobs)

print(f"=== BƯỚC 12: KẾT QUẢ KHỬ TRÙNG LẶP ===")
print(f"- Số lượng bản ghi trước khi khử trùng: {old_count:,} dòng")
print(f"- Số lượng bản ghi sau khi khử trùng : {new_count:,} dòng")
print(f"- Đã loại bỏ: {old_count - new_count:,} dòng trùng lặp")
jobs.head(5)


=== BƯỚC 12: KẾT QUẢ KHỬ TRÙNG LẶP ===
- Số lượng bản ghi trước khi khử trùng: 3,997 dòng
- Số lượng bản ghi sau khi khử trùng : 3,836 dòng
- Đã loại bỏ: 161 dòng trùng lặp


,job_id,company_name_raw,title_raw,description_raw,required_skills_raw,preferred_skills_raw,experience_raw,location_raw,salary_raw,employment_type_raw,...,experience_min,experience_max,location,city,salary_min,salary_max,employment_type,work_mode,deadline,is_active
474,topcv_2136013,CÔNG TY TNHH 1 THÀNH VIÊN CÔNG NGHỆ GIÁO DỤC S...,Kỹ Sư Hạ Tầng Công Nghệ Thông Tin,"1. Phụ trách lắp đặt, cấu hình và bảo trì các ...","1 năm kinh nghiệm chuyên môn, Xem tất cả việc ...",,1 năm,"KCN Phước Nam 2, Ninh Thuận cũ, Xã Thuận Nam, ...",Tới 40 triệu,Nhân viên,...,1.0,1.0,"KCN Phước Nam 2, Ninh Thuận cũ, Xã Thuận Nam, ...",Khác,NaN,40000000.0,Full-time,Onsite,2026-11-08,True
3915,topcv_1986323,CÔNG TY TNHH 1 THÀNH VIÊN ISOFHCARE,Chuyên Viên BA Nghiệp Vụ,Hỗ trợ Phân tích Yêu cầu: Làm việc dưới sự hướ...,"3 năm kinh nghiệm chuyên môn, Chuyên môn Công ...",,3 năm,"Tòa nhà One Mount. 458 Minh Khai, Phường Hai B...",Lên tới 25000000,Nhân viên,...,3.0,3.0,"Tòa nhà One Mount. 458 Minh Khai, Phường Hai B...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-11-25,True
2596,topcv_2077790,CÔNG TY TNHH 1 THÀNH VIÊN ISOFHCARE,Nhân Viên Phân Tích Yêu Cầu Ứng Dụng (BA),"Business Analyst chịu trách nhiệm thu thập, ph...","3 năm kinh nghiệm chuyên môn, Chuyên môn Công ...",,3 năm,"One Mount Building, 458 P. Minh Khai, Phường H...",Tới 25 triệu,Nhân viên,...,3.0,3.0,"One Mount Building, 458 P. Minh Khai, Phường H...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-10-22,False
3728,topcv_2339964,CÔNG TY TNHH 1 THÀNH VIÊN TRUYỀN THÔNG YUM NET...,Nhân Viên Designer Thiết Kế Parttime,"Thiết kế, quản lý và phát triển về mặt mỹ thuậ...","1 năm kinh nghiệm chuyên môn, Animation Design...",,1 năm,"23 Duy Tân,Cầu Giấy, Phường Cầu Giấy, Hà Nội",5000000 - 6000000,Nhân viên,...,1.0,1.0,"23 Duy Tân,Cầu Giấy, Phường Cầu Giấy, Hà Nội",Hà Nội,5000000.0,6000000.0,Part-time,Onsite,2026-11-23,True
3352,topcv_2334350,Công ty TNHH 1C Việt Nam,Chuyên Viên Kinh Doanh Phần Mềm Quản Trị,"Chủ động tìm kiếm, tiếp cận và phát triển khác...","2 năm kinh nghiệm chuyên môn, Kinh doanh phần ...",,2 năm,"Tầng 21 Tòa Century - 458 đường Minh Khai, Tim...",Thoả thuận,Nhân viên,...,2.0,2.0,"Tầng 21 Tòa Century - 458 đường Minh Khai, Tim...",Hà Nội,NaN,NaN,Full-time,Onsite,2026-11-18,True


## Bước 13: Xuất bảng kết quả cuối cùng ra CSV

Lựa chọn các cột chuẩn đầu ra và lưu vào thư mục `data/notebook/content_base/jobs_clean.csv`.

In [42]:
final_columns = [
    'job_id',
    'company_name',
    'title',
    'description_clean',
    'required_skills',
    'preferred_skills',
    'experience_min',
    'experience_max',
    'location',
    'city',
    'salary_min',
    'salary_max',
    'employment_type',
    'work_mode',
    'deadline',
    'is_active'
]

df_final = jobs[final_columns].copy()

df_final_csv = df_final.copy()
df_final_csv['required_skills'] = df_final_csv['required_skills'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
df_final_csv['preferred_skills'] = df_final_csv['preferred_skills'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

output_dir = os.path.abspath('../../data/notebook/content_base')
if not os.path.exists(output_dir):
    output_dir = os.path.abspath('data/notebook/content_base')
os.makedirs(output_dir, exist_ok=True)

output_csv_path = os.path.join(output_dir, 'jobs_clean.csv')
df_final_csv.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f" ĐÃ LƯU THÀNH CÔNG VÀO FILE:")
print(f" Đường dẫn: {output_csv_path}")
print(f" Số lượng bản ghi sạch: {len(df_final_csv):,} dòng x {df_final_csv.shape[1]} cột.")
df_final_csv.head(5)


 ĐÃ LƯU THÀNH CÔNG VÀO FILE:
 Đường dẫn: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\notebook\content_base\jobs_clean.csv
 Số lượng bản ghi sạch: 3,836 dòng x 16 cột.


,job_id,company_name,title,description_clean,required_skills,preferred_skills,experience_min,experience_max,location,city,salary_min,salary_max,employment_type,work_mode,deadline,is_active
474,topcv_2136013,1 THÀNH VIÊN CÔNG NGHỆ GIÁO DỤC SOTATEK,Kỹ Sư Hạ Tầng Công Nghệ Thông Tin,"1. Phụ trách lắp đặt, cấu hình và bảo trì các ...","Xem tất cả việc làm Kỹ thuật IT, Sản xuất, Tiế...",,1.0,1.0,"KCN Phước Nam 2, Ninh Thuận cũ, Xã Thuận Nam, ...",Khác,NaN,40000000.0,Full-time,Onsite,2026-11-08,True
3915,topcv_1986323,1 THÀNH VIÊN ISOFHCARE,Chuyên Viên BA Nghiệp Vụ,Hỗ trợ Phân tích Yêu cầu: Làm việc dưới sự hướ...,"3 năm kinh nghiệm chuyên môn, Chuyên môn Công ...",,3.0,3.0,"Tòa nhà One Mount. 458 Minh Khai, Phường Hai B...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-11-25,True
2596,topcv_2077790,1 THÀNH VIÊN ISOFHCARE,Nhân Viên Phân Tích Yêu Cầu Ứng Dụng (BA),"Business Analyst chịu trách nhiệm thu thập, ph...","3 năm kinh nghiệm chuyên môn, Chuyên môn Công ...",,3.0,3.0,"One Mount Building, 458 P. Minh Khai, Phường H...",Hà Nội,NaN,25000000.0,Full-time,Onsite,2026-10-22,False
3728,topcv_2339964,1 THÀNH VIÊN TRUYỀN THÔNG YUM NETWORK,Nhân Viên Designer Thiết Kế Parttime,"Thiết kế, quản lý và phát triển về mặt mỹ thuậ...","Animation Design, Marketing, Quảng cáo, Nam",,1.0,1.0,"23 Duy Tân,Cầu Giấy, Phường Cầu Giấy, Hà Nội",Hà Nội,5000000.0,6000000.0,Part-time,Onsite,2026-11-23,True
3352,topcv_2334350,1C,Chuyên Viên Kinh Doanh Phần Mềm Quản Trị,"Chủ động tìm kiếm, tiếp cận và phát triển khác...","Kinh doanh phần mềm, Direct Sales, B2B, Bán lẻ...",,2.0,2.0,"Tầng 21 Tòa Century - 458 đường Minh Khai, Tim...",Hà Nội,NaN,NaN,Full-time,Onsite,2026-11-18,True
